In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
import os
from IPython.display import Image as IPImage


In [ ]:
# Función para cargar la imagen desde el archivo
def cargar_imagen(change):
    archivo = list(upload_button.value.values())[0]
    img = Image.open(io.BytesIO(archivo['content']))
    img = np.array(img)
    if len(img.shape) == 3:  # Convertir a escala de grises si es RGB
        img = np.dot(img[...,:3], [0.2989, 0.5870, 0.1140])
    global imagen
    imagen = img
    mostrar_imagen(imagen, "Imagen Original")

# Función para mostrar la imagen en la interfaz
def mostrar_imagen(imagen, titulo):
    plt.figure(figsize=(6, 6))
    plt.imshow(imagen, cmap='gray')
    plt.axis('off')
    plt.title(titulo)
    plt.show()

# Función de preprocesamiento de la imagen para PCA
def preprocesar_imagen(imagen):
    imagen_normalizada = imagen / 255.0  # Normalizar entre 0 y 1
    return imagen_normalizada.reshape(-1, imagen.shape[1])  # Convertir en 2D

# PCA y compresión de la imagen
def comprimir_imagen_pca(imagen_2d, n_componentes=50):
    scaler = StandardScaler()
    imagen_2d_normalizada = scaler.fit_transform(imagen_2d)
    pca = PCA(n_components=n_componentes)
    componentes_principales = pca.fit_transform(imagen_2d_normalizada)
    return pca, componentes_principales, scaler

# Reconstrucción de la imagen comprimida
def reconstruir_imagen(componentes_principales, pca, scaler, imagen_2d_original):
    imagen_reconstruida_normalizada = pca.inverse_transform(componentes_principales)
    imagen_reconstruida = scaler.inverse_transform(imagen_reconstruida_normalizada)
    return imagen_reconstruida.reshape(imagen.shape)


In [ ]:
# Widget para subir la imagen
upload_button = widgets.FileUpload(accept="image/*", multiple=False)
upload_button.observe(cargar_imagen, names='value')

# Slider para seleccionar el número de componentes principales
slider_componentes = widgets.IntSlider(
    value=50,
    min=1,
    max=100,
    step=1,
    description='Componentes PCA:',
    continuous_update=False
)

# Botón para comprimir la imagen
button_comprimir = widgets.Button(description="Comprimir Imagen")

# Función que se ejecuta cuando se presiona el botón de comprimir
def on_comprimir_click(b):
    if 'imagen' in globals():
        imagen_2d = preprocesar_imagen(imagen)
        n_componentes = slider_componentes.value
        pca, componentes_principales, scaler = comprimir_imagen_pca(imagen_2d, n_componentes)
        imagen_reconstruida = reconstruir_imagen(componentes_principales, pca, scaler, imagen_2d)
        mostrar_imagen(imagen_reconstruida, f"Imagen Comprimida y Restaurada con {n_componentes} Componentes")
    else:
        print("Por favor, carga una imagen primero.")

button_comprimir.on_click(on_comprimir_click)

# Botón para descargar la imagen restaurada
button_descargar = widgets.Button(description="Descargar Imagen Restaurada")

# Función para descargar la imagen restaurada
def on_descargar_click(b):
    if 'imagen' in globals():
        imagen_2d = preprocesar_imagen(imagen)
        n_componentes = slider_componentes.value
        pca, componentes_principales, scaler = comprimir_imagen_pca(imagen_2d, n_componentes)
        imagen_reconstruida = reconstruir_imagen(componentes_principales, pca, scaler, imagen_2d)
        archivo = guardar_imagen(imagen_reconstruida)
        print(f"Imagen restaurada guardada como {archivo}")
    else:
        print("Por favor, carga una imagen primero.")

button_descargar.on_click(on_descargar_click)

# Función para guardar la imagen restaurada
def guardar_imagen(imagen_reconstruida, nombre_archivo='imagen_restaurada.jpg'):
    imagen_restaurada = Image.fromarray((imagen_reconstruida * 255).astype(np.uint8))
    imagen_restaurada.save(nombre_archivo)
    return nombre_archivo

# Mostrar los widgets en la interfaz
def mostrar_widgets():
    display(widgets.VBox([
        widgets.HTML("<h2>Aplicación de Compresión y Restauración de Imágenes con PCA</h2>"),
        widgets.HTML("<p>Sube tu imagen para comenzar el proceso de compresión y restauración.</p>"),
        upload_button,
        slider_componentes,
        button_comprimir,
        button_descargar
    ]))

mostrar_widgets()


In [ ]:
# Diseño y estilo básico de la aplicación

def mostrar_interfaz():
    # Titulo y descripción de la aplicación
    display(widgets.HTML("<h2>Compresión y Restauración de Imágenes usando PCA</h2>"))
    display(widgets.HTML("<p>Sube una imagen, selecciona el número de componentes principales, comprímela y restaura la imagen para verla.</p>"))
    
    # Mostrar los widgets para interactuar
    mostrar_widgets()

# Llamar a la función para mostrar la interfaz
mostrar_interfaz()
